# 10 · Contraste de hipótesis + conclusiones  *(sección 10 del script)*

**Qué hacemos:** llevamos a **tests estadísticos formales** las intuiciones que vinimos recogiendo en los notebooks 06-09:

| # | Hipótesis | Cómo se contrasta |
|---|---|---|
| **H1** | La demora en la entrega reduce la satisfacción del cliente | Mann-Whitney: `review_score` de "a tiempo" vs. "demorado" |
| **H2** | El contenido de las reseñas negativas está dominado por demoras/entrega | Evidencia cualitativa ya producida en el notebook 09 (nubes + ranking) |
| **H3** | La distancia al eje Sudeste se asocia a más tiempo de entrega y flete más caro | Tabla Sudeste vs. resto + Mann-Whitney sobre tiempo de entrega |
| **Adicional** | La distancia *real* (km) se asocia a mayor tiempo de entrega | Correlación de Spearman (variable continua) |

**Para qué:** separar lo que **es estadísticamente significativo** de lo que podría ser **azar** — un gráfico puede sugerir, pero el test es lo que permite afirmar.

## Método — ¿Por qué Mann-Whitney U y no un t-test?

- `review_score` **no es una variable continua normal**: es un puntaje ordinal (1-5 ★) con distribución fuertemente sesgada hacia el 5 (notebook 06). El t-test asume normalidad y homocedasticidad — acá no se cumplen.
- **Mann-Whitney U** (Wilcoxon rank-sum) es el test **no paramétrico** para comparar **dos grupos independientes**: no mira las medias, compara **rangos** — es robusto a la no-normalidad y a los outliers.
- **Hipótesis nula (H0):** las dos distribuciones son iguales (la diferencia es azar). **p-valor < 0.05** → rechazamos H0.
- Los tests acá son **de una cola** (`alternative`): cada uno apunta a la dirección concreta de la hipótesis ("a tiempo ≥ demorado", "Sudeste < resto").

> La **correlación de Spearman** (para la hipótesis adicional) es el análogo no paramétrico de Pearson: mide relación **monótona** (no necesariamente lineal) entre dos variables ordinales/continuas, con rango de -1 a +1 y su p-valor.

## Celda estándar: carga del artefacto

In [1]:
# Celda estándar (detallada en 01_configuracion_inicial.ipynb)
%matplotlib inline

import re
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

BASE = Path.cwd()
if not (BASE / "data").exists() and (BASE.parent / "data").exists():
    BASE = BASE.parent

CSV_UNIFICADO = BASE / "data" / "olist_dataset_unificado.csv"
PIPELINE_DIR = BASE / "data" / "pipeline"
FIGS_DIR = BASE / "figuras_eda"

OLIST_BLUE = "#0A4EE4"
OLIST_BLUE_DARK = "#0D366B"
OLIST_GRAY = "#52514E"
PALETA_CATEGORICA = ["#0A4EE4", "#eb6834", "#1baf7a", "#eda100",
                     "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SUDESTE = {"SP", "RJ", "MG", "ES"}

pd.set_option("display.max_columns", 50)

# --- Carga del artefacto del notebook 04 ---
COLUMNAS_FECHA = [
    "shipping_limit_date", "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

RUTA_FEATURES = PIPELINE_DIR / "02_df_features.csv"
if not RUTA_FEATURES.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_FEATURES}. Ejecutá primero los notebooks 02, 03 y 04."
    )

df = pd.read_csv(RUTA_FEATURES)
for col in COLUMNAS_FECHA:
    df[col] = pd.to_datetime(df[col], errors="coerce")
df["mes_compra"] = df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()

print(f"filas={df.shape[0]:,} columnas={df.shape[1]}")

filas=112,650 columnas=43


## H1 — ¿La demora reduce la satisfacción?

**Qué hacemos:**

1. Comparamos el `review_score` promedio entre pedidos **a tiempo** y **demorados**.
2. Corremos **Mann-Whitney U** con `alternative="greater"`: la hipótesis directional es que el puntaje de los **a tiempo es mayor** que el de los demorados (H0: son iguales).

**Para qué:** contrastar formalmente la hipótesis 1, yendo más allá del boxplot descriptivo del notebook 06.

In [2]:
print("review_score promedio por grupo:")
print(df.groupby("envio_demorado")["review_score"].mean().round(2))
print()

a_tiempo = df.loc[df["envio_demorado"] == 0, "review_score"].dropna()
demorado = df.loc[df["envio_demorado"] == 1, "review_score"].dropna()

u_stat, p_valor = stats.mannwhitneyu(a_tiempo, demorado, alternative="greater")
print(f"Tamaños: a tiempo n={len(a_tiempo):,} | demorado n={len(demorado):,}")
print(f"Mann-Whitney U: estadístico U={u_stat:,.0f}, p-valor={p_valor:.2e}")

review_score promedio por grupo:
envio_demorado
0.0    4.21
1.0    2.55
Name: review_score, dtype: float64

Tamaños: a tiempo n=100,849 | demorado n=8,520
Mann-Whitney U: estadístico U=657,437,282, p-valor=0.00e+00


### ✅ Veredicto H1: **confirmada**

**Insight:** el p-valor es **prácticamente cero**, muy por debajo del umbral de 0,05: la diferencia de satisfacción entre envíos a tiempo y demorados **no es casualidad** — es estadísticamente significativa, y en la dirección esperada (los a tiempo puntúan más alto). La logística no solo se atrasa: **le cuesta estrellas a la plataforma**.

## H3 — ¿El Sudeste tiene ventaja logística?

**Qué hacemos:**

1. Tabla comparativa **Sudeste vs. resto del país**: flete promedio, tiempo de entrega promedio y % de envíos demorados.
2. **Mann-Whitney U** sobre el tiempo de entrega con `alternative="less"`: la hipótesis es que el Sudeste entrega **más rápido** (menos días) que el resto.

**Para qué:** contrastar formalmente la hipótesis 3 — la que se venía anticipando en el mapa y el resumen por estado del notebook 08.

In [3]:
print("Comparación Sudeste vs. Resto del país:")
comparacion = df.groupby("region_sudeste").agg(
    flete_promedio=("freight_value", "mean"),
    tiempo_entrega_promedio=("tiempo_entrega_dias", "mean"),
    pct_demorado=("envio_demorado", "mean"),
).round(2)
comparacion["pct_demorado"] = (comparacion["pct_demorado"] * 100).round(1)
print(comparacion)
print()

sudeste = df.loc[df["region_sudeste"] == "Sudeste", "tiempo_entrega_dias"].dropna()
resto = df.loc[df["region_sudeste"] == "Resto del país", "tiempo_entrega_dias"].dropna()

u_stat, p_valor = stats.mannwhitneyu(sudeste, resto, alternative="less")
print(f"Tamaños: Sudeste n={len(sudeste):,} | Resto n={len(resto):,}")
print(f"Mann-Whitney U (tiempo de entrega, Sudeste < Resto): p-valor={p_valor:.2e}")

Comparación Sudeste vs. Resto del país:
                flete_promedio  tiempo_entrega_promedio  pct_demorado
region_sudeste                                                       
Resto del país           25.74                    15.94           9.0
Sudeste                  17.37                    10.22           7.0

Tamaños: Sudeste n=75,731 | Resto n=34,465
Mann-Whitney U (tiempo de entrega, Sudeste < Resto): p-valor=0.00e+00


### ✅ Veredicto H3: **confirmada**

**Insight:** los clientes **fuera del eje Sudeste pagan más flete, esperan más días y tienen mayor proporción de envíos demorados**, y el p-valor confirma que la diferencia en tiempo de entrega **no es azar**. La brecha geográfica es real — y recordemos (notebook 07) que **no** viene acompañada de una brecha de gasto: compran igual o más caro.

## Hipótesis adicional — ¿La distancia (km) explica los tiempos?

**Qué hacemos:** **correlación de Spearman** entre `distancia_km` y `tiempo_entrega_dias`, con su p-valor (`nan_policy="omit"` para ignorar los faltantes de coordenadas).

**Para qué:** complementar H3 con una **variable continua** en vez de la partición categórica Sudeste/resto: mide si *cada kilómetro extra* se asocia a más días de espera, con una relación que no necesita ser lineal.

In [4]:
rho, p_valor = stats.spearmanr(df["distancia_km"], df["tiempo_entrega_dias"], nan_policy="omit")
print(f"Correlación de Spearman (distancia_km vs. tiempo_entrega_dias): rho={rho:.2f}, p-valor={p_valor:.2e}")

Correlación de Spearman (distancia_km vs. tiempo_entrega_dias): rho=0.54, p-valor=0.00e+00


### ✅ Veredicto hipótesis adicional: **confirmada**

**Insight:** la correlación es **positiva** y el p-valor, prácticamente cero: **a mayor distancia cliente-vendedor, mayor tiempo de entrega**, y la relación no es casualidad. Esto confirma, con una métrica continua, que la **distancia geográfica real es un driver logístico medible** (coherente con el boxplot y la correlación del notebook 07).

## Resumen de hipótesis

| Hipótesis | Evidencia | p-valor | Veredicto |
|---|---|---|---|
| **H1** — La demora reduce la satisfacción | Mann-Whitney (a tiempo > demorado) | ≈ 0 | ✅ **confirmada** |
| **H2** — Las reseñas negativas hablan de demoras | Nubes de palabras + top-15 frecuencias (nb 09) | — (cualitativa) | ✅ **evidenciada** |
| **H3** — Fuera del Sudeste: más días y más flete | Mann-Whitney (Sudeste < resto) | ≈ 0 | ✅ **confirmada** |
| **Adicional** — Más km ⇒ más días | Spearman (rho > 0) | ≈ 0 | ✅ **confirmada** |

## Conclusiones principales del EDA

1. **La logística es el problema central de la experiencia de Olist:** casi 1 de cada 10 pedidos entregados llega tarde (notebook 04), y eso se traduce directamente en menos estrellas (H1).
2. **La distancia geográfica real es un driver medible:** cientos de km de mediana cliente-vendedor, correlación positiva con el tiempo (Spearman) y brecha Sudeste/resto confirmada (H3) — pese a lo cual **el gasto por fuera del Sudeste es igual o mayor**: el cliente periférico paga más flete y espera más por un ticket similar.
3. **Los clientes lo dicen con palabras:** el vocabulario de las reseñas negativas está dominado por "prazo/atraso/entrega" (H2) — la queja es logística, no del producto.
4. **El negocio está razonablemente repartido:** la facturación no se concentra en 2-3 categorías (Pareto del nb 07), y los vendedores con peor desempeño logístico **no son los más grandes** — la mejora debe ser pareja y transversal.
5. **Volumen y precio son ejes distintos:** las categorías top en ítems son de ticket bajo; las de ticket alto mueven menos volumen (nb 06/07).

## Próximos pasos (fuera de este pipeline)

- `modelos_predictivos.py` — **clasificación** (predecir reseñas negativas/neutrales/positivas con Decision Tree, Random Forest y XGBoost) + **clustering** de vendedores (K-Means con codo/silhouette).
- `app.py` — **dashboard en Streamlit** que presenta todo el análisis y los modelos.
- `Informe_Funcional.docx` — la explicación de negocio completa, conclusiones y recomendaciones.

## Verificación final — Artefactos del pipeline

**Qué hacemos:** listamos los archivos que dejó el pipeline completo (datos + figuras).

**Para qué:** comprobar de un vistazo que la cadena 02 → 04 se ejecutó bien y qué imágenes quedaron archivadas para el informe/dashboard.

In [5]:
print("Datos:")
for ruta, origen in [
    (CSV_UNIFICADO, "notebook 02 - unificación"),
    (PIPELINE_DIR / "01_df_limpio.csv", "notebook 03 - limpieza"),
    (PIPELINE_DIR / "02_df_features.csv", "notebook 04 - features"),
]:
    if ruta.exists():
        mb = ruta.stat().st_size / 1e6
        print(f"  [OK]    {ruta.relative_to(BASE)} ({mb:.1f} MB)  <- {origen}")
    else:
        print(f"  [FALTA] {ruta.relative_to(BASE)}  <- {origen}")

figs = sorted(FIGS_DIR.glob("nb*.png"))
print(f"\nFiguras de los notebooks: {len(figs)} en {FIGS_DIR.relative_to(BASE)}/")
for f in figs:
    print("   ", f.name)

Datos:
  [OK]    data\olist_dataset_unificado.csv (61.8 MB)  <- notebook 02 - unificación
  [OK]    data\pipeline\01_df_limpio.csv (56.2 MB)  <- notebook 03 - limpieza
  [OK]    data\pipeline\02_df_features.csv (65.6 MB)  <- notebook 04 - features

Figuras de los notebooks: 18 en figuras_eda/
    nb01_01.png
    nb06_01.png
    nb06_02.png
    nb06_03.png
    nb06_04.png
    nb06_05.png
    nb06_06.png
    nb06_07.png
    nb06_08.png
    nb07_01.png
    nb07_02.png
    nb07_03.png
    nb07_04.png
    nb07_05.png
    nb07_06.png
    nb07_07.png
    nb09_01.png
    nb09_02.png


**Insight:** con los tres CSV de `data/` y las figuras `nbXX_*.png` en `figuras_eda/`, el EDA queda **cerrado y reproducible**: cualquier persona puede rehacerlo desde cero ejecutando los notebooks 00 → 10 en orden.

> 🏁 **Fin del pipeline de EDA.** El modelado predictivo (clasificación de reseñas, segmentación de vendedores) queda para la etapa siguiente, una vez validado este análisis exploratorio — tal como lo describe el `README.md` del proyecto.